In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}

ELECTRODE_TO_REGION = {
    electrode: region
    for region, electrodes in REGION_TO_ELECTRODES.items()
    for electrode in electrodes
}

REGION_TO_LABEL = {
    'Middle_Finger': 0,
    'Hand': 1,
    'Forearm': 2,
    'Arm': 3
}

LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

print("Region-to-Electrode Mapping:")
for region, electrodes in REGION_TO_ELECTRODES.items():
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {electrodes} ({len(electrodes)} electrodes)")

In [ ]:
BIDS_ROOT = Path(r"../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"

subjects = ["sub-p0002"]
session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs_per_subject = 4

HRF_DELAY = 6.0
WINDOW = 1

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])
print(f"Subject: {subjects[0]}")
print(f"Runs: {n_runs_per_subject}")
print(f"TR: {tr} s")

In [ ]:
RESULTS_DIR = Path("results/4_Classes_ANN_ICA")
FIGURES_DIR = RESULTS_DIR / "figures"
MODELS_DIR = RESULTS_DIR / "models"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_LOG = RESULTS_DIR / "4_Classes_ANN_ICA_analysis_log.txt"
log_file = open(OUTPUT_LOG, 'w', encoding='utf-8')
all_experiments_results = []

print(f"Results directory : {RESULTS_DIR.resolve()}")
print(f"Figures directory : {FIGURES_DIR.resolve()}")
print(f"Models directory  : {MODELS_DIR.resolve()}")

In [ ]:
all_events = []
subject = subjects[0]
for run in range(1, n_runs_per_subject + 1):
    events_path = BIDS_ROOT / subject / session / "func" / f"{subject}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(events_path, sep='\t')
    df['subject'] = subject
    df['run'] = run
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)

print(f"Total events: {len(events_df)}")
print(f"Stimulation events: {len(stim_events)}")
print(f"\nSamples per run:")
print(stim_events.groupby('run').size().to_dict())
print(f"\nSamples per region:")
for region in ['Middle_Finger', 'Hand', 'Forearm', 'Arm']:
    count = stim_events['region'].value_counts().get(region, 0)
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {count} samples")

In [ ]:
from nilearn.image import load_img, index_img, resample_to_img, new_img_like
from nilearn import plotting
from nilearn.maskers import NiftiMasker
from nilearn.datasets import fetch_atlas_destrieux_2009

first_run_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii")
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)

atlas = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
mask_data = np.isin(atlas_data, s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')

masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)

print(f"Reference image shape: {first_run_img.shape}")
print(f"S1 voxels: {int(np.sum(s1_mask_resampled.get_fdata() > 0))}")
print('Selected atlas regions:')
for i in s1_indices:
    print(f"  - {atlas.labels[i]}")

display = plotting.plot_roi(s1_mask_resampled, bg_img=ref_img,
                            title='S1 ROI Mask (Left G_postcentral)',
                            display_mode='ortho', cut_coords=(0, -20, 60))
display.savefig(FIGURES_DIR / 's1_mask.png', dpi=150)
display.close()
print(f"\nS1 mask saved to: {FIGURES_DIR / 's1_mask.png'}")

In [ ]:
X_list = []
y_list = []
run_labels = []

for run in range(1, n_runs_per_subject + 1):
    bold_path = (DERIVATIVES / subjects[0] / session / "func" /
                 f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii")
    bold_img = load_img(str(bold_path))
    print(f"Run {run}...")
    run_events = stim_events[stim_events['run'] == run]
    for _, event in run_events.iterrows():
        onset = event["onset"]
        region = event["region"]
        target_time = onset + HRF_DELAY
        vol_indx = int(np.round(target_time / tr))

        # Average of 3 volumes around the HRF peak
        vol_indices = [vol_indx - WINDOW, vol_indx, vol_indx + WINDOW]
        if all(0 <= v < bold_img.shape[3] for v in vol_indices):
            vol_data_list = [masker.transform(index_img(bold_img, v)) for v in vol_indices]
            vol_data_list = [vd[0] if len(vd.shape) == 2 else vd for vd in vol_data_list]
            vol_data_avg = np.mean(vol_data_list, axis=0)
            X_list.append(vol_data_avg)
            y_list.append(region)
            run_labels.append(run)

X = np.array(X_list)
y = np.array(y_list)
run_labels = np.array(run_labels)
y_encoded = np.array([REGION_TO_LABEL[region] for region in y])

print(f"\nFeature matrix shape: {X.shape}")
print(f"Label distribution:")
for region, label in REGION_TO_LABEL.items():
    print(f"  {region} (Class {label}): {np.sum(y_encoded == label)} samples")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_dict = {i: w for i, w in enumerate(class_weights)}
unique_labels = sorted(np.unique(y_encoded))

print("Class weights:")
for label, weight in class_weights_dict.items():
    print(f"  Class {label} ({LABEL_TO_REGION[label]}): weight={weight:.3f}, n={np.sum(y_encoded == label)}")

In [ ]:
from torch.nn import Module, Linear, BatchNorm1d, ReLU, Dropout, Sequential

class SomatotopicANN(Module):
    def __init__(self, input_size, hidden1, hidden2=None, num_classes=4, dropout_rate=0.3):
        super(SomatotopicANN, self).__init__()
        layers = [
            Linear(input_size, hidden1),
            BatchNorm1d(hidden1),
            ReLU(),
            Dropout(dropout_rate),
        ]
        if hidden2 is not None:
            layers += [
                Linear(hidden1, hidden2),
                BatchNorm1d(hidden2),
                ReLU(),
                Dropout(dropout_rate),
            ]
        out_features = hidden2 if hidden2 is not None else hidden1
        layers.append(Linear(out_features, num_classes))
        self.network = Sequential(*layers)

    def forward(self, x):
        return self.network(x)


In [ ]:
hidden1_options      = [64, 128, 256]
layer_combinations_2 = [(256, 64), (128, 32), (64, 16)]
wd_options           = [0.0001, 0.001, 0.01]
dr_options           = [0.2, 0.3, 0.4]
n_comp_options       = [10, 20, 30, 50]

configs_1l_none, configs_2l_none = [], []
configs_1l_pca,  configs_2l_pca  = [], []
configs_1l_ica,  configs_2l_ica  = [], []

for h1 in hidden1_options:
    for wd in wd_options:
        for dr in dr_options:
            base = {'hidden1': h1, 'hidden2': None, 'weight_decay': wd, 'dropout_rate': dr}
            configs_1l_none.append({**base, 'reduction': 'none', 'n_components': None, 'architecture': '1-layer'})
            for n in n_comp_options:
                configs_1l_pca.append({**base, 'reduction': 'pca', 'n_components': n, 'architecture': f'1-layer-PCA{n}'})
                configs_1l_ica.append({**base, 'reduction': 'ica', 'n_components': n, 'architecture': f'1-layer-ICA{n}'})

for h1, h2 in layer_combinations_2:
    for wd in wd_options:
        for dr in dr_options:
            base = {'hidden1': h1, 'hidden2': h2, 'weight_decay': wd, 'dropout_rate': dr}
            configs_2l_none.append({**base, 'reduction': 'none', 'n_components': None, 'architecture': '2-layer'})
            for n in n_comp_options:
                configs_2l_pca.append({**base, 'reduction': 'pca', 'n_components': n, 'architecture': f'2-layer-PCA{n}'})
                configs_2l_ica.append({**base, 'reduction': 'ica', 'n_components': n, 'architecture': f'2-layer-ICA{n}'})

all_configs = (configs_1l_none + configs_2l_none +
               configs_1l_pca  + configs_2l_pca  +
               configs_1l_ica  + configs_2l_ica)

print(f"Total experiments : {len(all_configs)}")
print(f"  - 1-layer  (no reduction): {len(configs_1l_none)}")
print(f"  - 2-layer  (no reduction): {len(configs_2l_none)}")
print(f"  - 1-layer  (PCA): {len(configs_1l_pca)}")
print(f"  - 2-layer  (PCA): {len(configs_2l_pca)}")
print(f"  - 1-layer  (ICA): {len(configs_1l_ica)}")
print(f"  - 2-layer  (ICA): {len(configs_2l_ica)}")


In [ ]:
import time
import copy

def calculate_specificity(y_true, y_pred, class_label):
    y_true_binary = (y_true == class_label).astype(int)
    y_pred_binary = (y_pred == class_label).astype(int)
    tn = np.sum((y_true_binary == 0) & (y_pred_binary == 0))
    fp = np.sum((y_true_binary == 0) & (y_pred_binary == 1))
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

learning_rate = 0.001
target = 70.0
max_no_improvement = 1000
batch_size = 32
num_classes = len(unique_labels)
input_size = X.shape[1]

print(f"Training config: lr={learning_rate}, target={target}%, max_no_improvement={max_no_improvement}, batch_size={batch_size}")
print(f"Input size (raw): {input_size} voxels")

In [ ]:
from torch import FloatTensor, LongTensor, manual_seed, no_grad, max as torch_max, save
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import CrossEntropyLoss
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FastICA, PCA
import pickle

print("Starting Multi-Model Training (1-layer + 2-layer | none / PCA / ICA)...\n")
log_file.write("MULTI-MODEL TRAINING LOG — 1L+2L | none/PCA/ICA\n\n")

all_experiments_results = []

for exp_idx, config in enumerate(all_configs, 1):

    h2_str = f"->{config['hidden2']}" if config['hidden2'] is not None else ""
    print(f"\n{'='*80}")
    print(f"EXPERIMENT {exp_idx}/{len(all_configs)} | {config['architecture']} | "
          f"h={config['hidden1']}{h2_str} | wd={config['weight_decay']} | dr={config['dropout_rate']} | "
          f"reduction={config['reduction']} n={config['n_components']}")
    print(f"{'='*80}")
    log_file.write(f"\nEXPERIMENT {exp_idx}/{len(all_configs)} | Config: {config}\n")
    log_file.flush()

    hidden1      = config['hidden1']
    hidden2      = config['hidden2']
    weight_decay = config['weight_decay']
    dropout_rate = config['dropout_rate']
    reduction    = config['reduction']
    n_components = config['n_components']

    criterion    = CrossEntropyLoss(weight=FloatTensor(class_weights))
    fold_results = []
    start_time   = time.time()

    # Track the single best fold (by balanced accuracy) for saving
    overall_best_bal_acc  = 0.0
    overall_best_state    = None
    overall_best_scaler   = None
    overall_best_reducer  = None
    overall_best_fold     = None

    for test_run in range(1, n_runs_per_subject + 1):
        print(f"  Fold {test_run}/4...", end=' ')

        train_mask = run_labels != test_run
        test_mask  = run_labels == test_run

        X_train, X_test = X[train_mask], X[test_mask]
        y_train, y_test = y_encoded[train_mask], y_encoded[test_mask]

        scaler         = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)

        if reduction == 'pca':
            reducer       = PCA(n_components=n_components, random_state=RANDOM_SEED)
            X_train_proc  = reducer.fit_transform(X_train_scaled)
            X_test_proc   = reducer.transform(X_test_scaled)
            current_input_size = n_components
        elif reduction == 'ica':
            reducer       = FastICA(n_components=n_components, random_state=RANDOM_SEED,
                                    max_iter=500, whiten='unit-variance')
            X_train_proc  = reducer.fit_transform(X_train_scaled)
            X_test_proc   = reducer.transform(X_test_scaled)
            current_input_size = n_components
        else:
            reducer       = None
            X_train_proc  = X_train_scaled
            X_test_proc   = X_test_scaled
            current_input_size = input_size

        X_train_tensor = FloatTensor(X_train_proc)
        y_train_tensor = LongTensor(y_train)
        X_test_tensor  = FloatTensor(X_test_proc)
        y_test_tensor  = LongTensor(y_test)

        train_loader = DataLoader(
            TensorDataset(X_train_tensor, y_train_tensor),
            batch_size=batch_size, shuffle=True
        )

        manual_seed(RANDOM_SEED)
        model     = SomatotopicANN(current_input_size, hidden1, hidden2, num_classes, dropout_rate)
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=50, min_lr=1e-6)

        epoch                  = 0
        best_balanced_accuracy = 0.0
        epochs_no_improvement  = 0
        best_model_state       = None

        while True:
            model.train()
            for batch_X, batch_y in train_loader:
                outputs = model(batch_X)
                loss    = criterion(outputs, batch_y)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with no_grad():
                test_outputs      = model(X_test_tensor)
                _, test_predicted = torch_max(test_outputs.data, 1)

            test_bal_acc = balanced_accuracy_score(y_test_tensor.numpy(), test_predicted.numpy()) * 100
            scheduler.step(test_bal_acc)

            if test_bal_acc >= target:
                best_model_state = copy.deepcopy(model.state_dict())
                break

            if test_bal_acc > best_balanced_accuracy:
                best_balanced_accuracy = test_bal_acc
                best_model_state       = copy.deepcopy(model.state_dict())
                epochs_no_improvement  = 0
            else:
                epochs_no_improvement += 1

            if epochs_no_improvement >= max_no_improvement:
                break

            epoch += 1

        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        model.eval()
        with no_grad():
            _, final_predictions = torch_max(model(X_test_tensor).data, 1)

        y_pred            = final_predictions.numpy()
        fold_accuracy     = accuracy_score(y_test, y_pred)
        fold_bal_accuracy = balanced_accuracy_score(y_test, y_pred)
        cm_fold           = confusion_matrix(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )
        specificities = [calculate_specificity(y_test, y_pred, c) for c in range(4)]

        fold_results.append({
            'fold': test_run,
            'accuracy': fold_accuracy,
            'balanced_accuracy': fold_bal_accuracy,
            'epochs': epoch + 1,
            'confusion_matrix': cm_fold,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'specificity': np.array(specificities)
        })
        print(f"Acc: {fold_accuracy*100:.2f}% | Bal.Acc: {fold_bal_accuracy*100:.2f}% ({epoch+1} epochs)")

        # Track best fold for saving (best balanced accuracy across all folds)
        if fold_bal_accuracy > overall_best_bal_acc:
            overall_best_bal_acc = fold_bal_accuracy
            overall_best_state   = copy.deepcopy(model.state_dict())
            overall_best_scaler  = copy.deepcopy(scaler)
            overall_best_reducer = copy.deepcopy(reducer)
            overall_best_fold    = test_run

    # ── Aggregate fold results ──────────────────────────────────────────────
    fold_accuracies     = [r['accuracy']         for r in fold_results]
    fold_bal_accuracies = [r['balanced_accuracy'] for r in fold_results]
    mean_acc     = np.mean(fold_accuracies)
    std_acc      = np.std(fold_accuracies)
    mean_bal_acc = np.mean(fold_bal_accuracies)
    std_bal_acc  = np.std(fold_bal_accuracies)
    elapsed_time = time.time() - start_time
    cm_total     = sum(r['confusion_matrix'] for r in fold_results)

    mean_precision_per_class   = np.mean([r['precision']   for r in fold_results], axis=0)
    mean_recall_per_class      = np.mean([r['recall']      for r in fold_results], axis=0)
    mean_f1_per_class          = np.mean([r['f1']          for r in fold_results], axis=0)
    mean_specificity_per_class = np.mean([r['specificity'] for r in fold_results], axis=0)

    macro_precision   = np.mean(mean_precision_per_class)
    macro_recall      = np.mean(mean_recall_per_class)
    macro_f1          = np.mean(mean_f1_per_class)
    macro_specificity = np.mean(mean_specificity_per_class)

    total_support        = cm_total.sum(axis=1)
    weighted_precision   = np.average(mean_precision_per_class,   weights=total_support)
    weighted_recall      = np.average(mean_recall_per_class,      weights=total_support)
    weighted_f1          = np.average(mean_f1_per_class,          weights=total_support)
    weighted_specificity = np.average(mean_specificity_per_class, weights=total_support)

    print(f"  Mean Accuracy     : {mean_acc*100:.2f}% +/- {std_acc*100:.2f}%")
    print(f"  Mean Bal. Accuracy: {mean_bal_acc*100:.2f}% +/- {std_bal_acc*100:.2f}% | Time: {elapsed_time:.1f}s")
    print(f"  Best fold         : {overall_best_fold} (bal.acc={overall_best_bal_acc*100:.2f}%)")

    # ── Save best-fold model, scaler, and reducer ────────────────────────────
    red_str        = f"_{reduction}{n_components}" if reduction != 'none' else ""
    h_str          = f"h{hidden1}-{hidden2}" if hidden2 is not None else f"h{hidden1}"
    model_filename  = f"model_{h_str}_wd{weight_decay}_dr{dropout_rate}{red_str}.pt"
    cm_filename     = model_filename.replace('.pt', '_confusion_matrix.npy')
    scaler_filename = model_filename.replace('.pt', '_scaler.pkl')

    save(overall_best_state, MODELS_DIR / model_filename)
    np.save(MODELS_DIR / cm_filename, cm_total)
    with open(MODELS_DIR / scaler_filename, 'wb') as _f:
        pickle.dump(overall_best_scaler, _f)

    if overall_best_reducer is not None:
        reducer_filename = model_filename.replace('.pt', '_reducer.pkl')
        with open(MODELS_DIR / reducer_filename, 'wb') as _f:
            pickle.dump(overall_best_reducer, _f)
        reducer_file_saved = reducer_filename
    else:
        reducer_file_saved = 'None'

    # ── Append result ────────────────────────────────────────────────────────
    all_experiments_results.append({
        'experiment_id': exp_idx,
        'architecture': config['architecture'],
        'hidden1': hidden1,
        'hidden2': hidden2 if hidden2 is not None else 'None',
        'weight_decay': weight_decay,
        'dropout_rate': dropout_rate,
        'reduction': reduction,
        'n_components': n_components if n_components is not None else 'None',
        'learning_rate': learning_rate,
        'mean_accuracy': mean_acc,
        'std_accuracy': std_acc,
        'mean_balanced_accuracy': mean_bal_acc,
        'std_balanced_accuracy': std_bal_acc,
        'best_fold': overall_best_fold,
        'best_fold_bal_accuracy': overall_best_bal_acc,
        'macro_sensitivity': macro_recall,
        'macro_specificity': macro_specificity,
        'macro_precision': macro_precision,
        'macro_f1': macro_f1,
        'weighted_sensitivity': weighted_recall,
        'weighted_specificity': weighted_specificity,
        'weighted_precision': weighted_precision,
        'weighted_f1': weighted_f1,
        'per_class_sensitivity': str(mean_recall_per_class.tolist()),
        'per_class_specificity': str(mean_specificity_per_class.tolist()),
        'per_class_precision': str(mean_precision_per_class.tolist()),
        'per_class_f1': str(mean_f1_per_class.tolist()),
        'total_epochs': sum(r['epochs'] for r in fold_results),
        'training_time_s': elapsed_time,
        'model_file': model_filename,
        'scaler_file': scaler_filename,
        'reducer_file': reducer_file_saved,
        'confusion_matrix_file': cm_filename
    })

    pd.DataFrame(all_experiments_results).to_csv(RESULTS_DIR / 'all_experiments.csv', index=False)
    log_file.write(f"Mean Accuracy: {mean_acc*100:.2f}% +/- {std_acc*100:.2f}%\n")
    log_file.write(f"Mean Balanced Accuracy: {mean_bal_acc*100:.2f}% +/- {std_bal_acc*100:.2f}%\n")
    log_file.write(f"Macro Sensitivity: {macro_recall:.4f}, Macro Specificity: {macro_specificity:.4f}\n")
    log_file.write(f"Best fold: {overall_best_fold} | Model: {model_filename}\n\n")
    log_file.flush()

print(f"\n{'='*80}")
print(f"ALL {len(all_configs)} EXPERIMENTS COMPLETE")
print(f"{'='*80}")
log_file.write("\nALL EXPERIMENTS COMPLETE\n")
log_file.flush()


In [ ]:
log_file.close()
print(f"Results saved to: {RESULTS_DIR.resolve()}")
print(f"  - Log      : {OUTPUT_LOG}")
print(f"  - CSV      : {RESULTS_DIR / 'all_experiments.csv'}")
print(f"  - Models   : {MODELS_DIR}")

## Results Summary

In [ ]:
results_df = pd.read_csv(RESULTS_DIR / 'all_experiments.csv')
results_df_sorted = results_df.sort_values('mean_balanced_accuracy', ascending=False)

print(f"Total experiments: {len(results_df)}")
print(f"\nTop 10 models by mean balanced accuracy:")
print(results_df_sorted[[
    'architecture', 'hidden1', 'hidden2', 'weight_decay', 'dropout_rate',
    'reduction', 'n_components', 'mean_balanced_accuracy', 'mean_accuracy'
]].head(10).to_string(index=False))

best = results_df_sorted.iloc[0]
print(f"\nBest model:")
print(f"  Architecture     : {best['architecture']}")
print(f"  Hidden layers    : {best['hidden1']} -> {best['hidden2']}")
print(f"  Weight decay     : {best['weight_decay']}")
print(f"  Dropout          : {best['dropout_rate']}")
print(f"  Reduction        : {best['reduction']} (n={best['n_components']})")
print(f"  Balanced accuracy: {best['mean_balanced_accuracy']*100:.2f}% +/- {best['std_balanced_accuracy']*100:.2f}%")
print(f"  Accuracy         : {best['mean_accuracy']*100:.2f}%")

results_df_sorted.head(15).to_csv(RESULTS_DIR / 'top15_models.csv', index=False)
print(f"\nTop 15 saved to: {RESULTS_DIR / 'top15_models.csv'}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot: balanced accuracy by reduction type
all_reductions = ['none', 'pca', 'ica']
order = [r for r in all_reductions if r in results_df['reduction'].values]
sns.boxplot(data=results_df, x='reduction', y='mean_balanced_accuracy', order=order, ax=axes[0])
axes[0].axhline(y=0.25, color='red', linestyle='--', label='Chance (25%)')
axes[0].set_title('Balanced Accuracy by Reduction Type')
axes[0].set_xlabel('Reduction')
axes[0].set_ylabel('Mean CV Balanced Accuracy')
axes[0].legend()

# Confusion matrix of the best model (sorted by balanced accuracy)
best_cm = np.load(MODELS_DIR / best['confusion_matrix_file'])
class_names = [LABEL_TO_REGION[i] for i in range(4)]
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title(f'Best Model Confusion Matrix\n({best["architecture"]}, {best["mean_balanced_accuracy"]*100:.1f}% bal.acc.)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'accuracy_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved to: {FIGURES_DIR / 'accuracy_summary.png'}")